In [6]:
# --- repo bootstrap: make src/ importable and run from repo root (works wherever the kernel starts) ---
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
sys.path.insert(0, str(_ROOT / 'schemas'))   # hand-authored reference schemas
os.chdir(_ROOT)

# Aave V3.1 — data transformation

**Step 1: load & display.** Reads the *latest* versioned CSV per table from
`query_result_data/` and displays it. The raw extracts are treated as
**read-only** — nothing here writes back to that folder.

All loading logic lives in `transform.py` (libraries it imports: `pandas`,
stdlib `re`/`pathlib`, and `data_validation` for the shared query-id → label map).
Edit `TABLE` to choose which table to display; `PREVIEW_ROWS` sets the preview size.

In [7]:
# Cell 1 — imports + notebook-level settings
import pandas as pd
from IPython.display import display
import transform as tf
import data_validation as dv
import adv_validation as adv
import schema_check as sc
from process_schema import SCHEMAS as PROCESS_SCHEMAS      # what the raw tables must look like
from transform_schema import SCHEMAS as TRANSFORM_SCHEMAS  # what this notebook must produce

PREVIEW_ROWS = 10  # rows shown in the inline preview (notebook-only setting)


In [8]:
# Cell 2 — the timelines available to run on. One row per extraction window, built from what
# each file's ROWS say it covers (year+month), not from its name: inside one window the
# tables end on different days, so a day-level match would split one window into three.
display(tf.timelines())


,batch,tables,first_date,last_date,select_with
0,2024-04 → 2025-03,9,2024-04-01,2025-03-31,'2024-04'
1,2025-04 → 2026-03,9,2025-04-01,2026-03-31,'2025-04'


In [9]:
# Cell 3 — PICK ONE. Paste a value from `select_with` above; None = the newest timeline.
#   BATCH = "2024-04"              a month, when it names exactly one window
#   BATCH = ("2024-04","2025-03")  the full span, always unambiguous
#   BATCH = None                   newest
BATCH = None

In [10]:
# use_batch resolves that choice ONCE and hands back a run that can see ONLY that window's
# files. Every load below goes through `run`, so no other timeline's table can enter this run
# — a label outside the batch raises instead of quietly resolving to another window, which is
# what a forgotten `batch=` argument used to do. The decimal references come along because
# they carry no time column and belong to every window.
# Every raw table is schema-checked at ITS OWN load cell below rather than in one call here:
# this notebook loads progressively across a dozen cells, so a single up-front check would
# either re-read 163 MB or run after the frames had already been used. Checking at the load
# site fails at the exact cell that read the bad table.
run = tf.use_normalized_batch(BATCH)
print(run)
display(run.tables())     # exactly the files this run may read — nothing else is reachable

<BatchRun 2025-04 → 2026-03 · 16 tables available · 0 loaded · from query_result_data/>


,label,file
0,borrow_repay,borrow_repay_7798273_2025-04-01_2026-03-31.csv
1,collateral_toggle,collateral_toggle_7798372_2025-04-01_2026-03-3...
2,flashloan,flashloan_7798349_2025-04-01_2026-03-31.csv
3,liquidation,liquidation_7798339_2025-04-01_2026-03-31.csv
4,oracle_price_usd_eth_weth_6h,oracle_price_usd_eth_weth_6h_8403530_2025-04-0...
5,reserve_config,reserve_config_7804264_asof_2026-03-31.csv
6,reserve_state_rates,reserve_state_rates_7711042_2025-04-01_2026-03...
7,supply_withdraw,supply_withdraw_8595322_2025-04-01_2026-03-31.csv
8,user_account,user_account_7798351_2025-04-01_2026-03-31.csv
9,decimal_reference_part1,decimal_reference_part1_7711171_20260623T19032...


In [11]:
# Cell 4 — cross-window check: label x timeline, blank = that table is MISSING from that
# window. (tf.table_coverage() gives the per-file detail, incl. which labels had to be
# recovered from the filename because TABLE_LABELS has no entry for their query id.)
display(tf.batch_matrix())

,2024-04 → 2025-03,2025-04 → 2026-03,static
borrow_repay,borrow_repay_7798273_2024-04-01_2025-03-31.csv,borrow_repay_7798273_2025-04-01_2026-03-31.csv,
collateral_toggle,collateral_toggle_7798372_2024-04-01_2025-03-3...,collateral_toggle_7798372_2025-04-01_2026-03-3...,
decimal_reference_part1,,,decimal_reference_part1_7711171_20260623T19032...
decimal_reference_part2,,,decimal_reference_part2_7711265_20260623T19175...
decimal_reference_part3,,,decimal_reference_part3_7711276_20260623T19230...
decimal_reference_part4,,,decimal_reference_part4_7711290_20260623T19231...
decimal_reference_part5,,,decimal_reference_part5_7711298_20260623T19232...
decimal_reference_part6,,,decimal_reference_part6_7711304_20260623T19232...
decimal_reference_part9_collateral_toggle,,,decimal_reference_part9_collateral_toggle_7711...
flashloan,flashloan_7798349_2024-04-01_2025-03-31.csv,flashloan_7798349_2025-04-01_2026-03-31.csv,


In [12]:
# Cell 3 — load one table (read-only) and display it
TABLE = "supply_withdraw"  # <-- a label from run.tables(); the run owns which file that is

df_0_supply_withdraw = run.load(TABLE)
sc.check_frame(df_0_supply_withdraw, PROCESS_SCHEMAS[TABLE], table=TABLE, stage="transform:input")
print(f"{TABLE}: {df_0_supply_withdraw.shape[0]} rows x {df_0_supply_withdraw.shape[1]} cols")
display(df_0_supply_withdraw.dtypes.to_frame("dtype"))
display(df_0_supply_withdraw.head(PREVIEW_ROWS))

supply_withdraw: 84609 rows x 12 cols


,dtype
time_bucket,object
asset,object
asset_symbol,object
supply_amount_raw,object
withdrawal_amount_raw,object
net_supply_flow_raw,object
supply_tx_count,int64
withdrawal_tx_count,int64
unique_suppliers,int64
unique_withdraw_users,int64


,time_bucket,asset,asset_symbol,supply_amount_raw,withdrawal_amount_raw,net_supply_flow_raw,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,latest_supply_block,latest_withdraw_block
0,2025-04-01 02:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,80937905573546271332,3352481644830690207,77585423928715581125,4,1,3,1,22170797.0,22170797.0
1,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,720471761,246464895,474006866,12,2,5,1,22170878.0,22170544.0
2,2025-04-01 02:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,20000000000000000000,255756260689565740664,-235756260689565740664,1,1,1,1,22170813.0,22170460.0
3,2025-04-01 02:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,101772080173949086951,NaN,101772080173949086951,2,0,2,0,22170680.0,NaN
4,2025-04-01 02:00:00.000 UTC,0x5f98805a4e8be255a32880fdec7f6728c6568ba0,LUSD,18242549021403584259,100920688337165907518,-82678139315762323259,1,1,1,1,22170885.0,22170883.0
5,2025-04-01 02:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,NaN,390160229791931868145049,-390160229791931868145049,0,1,0,1,NaN,22170539.0
6,2025-04-01 02:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,584566561797,543123699307,41442862490,1,5,1,1,22170836.0,22170922.0
7,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,22598482970008222347,26951146895841088925,-4352663925832866578,3,1,2,1,22170833.0,22170805.0
8,2025-04-01 02:00:00.000 UTC,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,AAVE,83734342036557266,1923275558078465767,-1839541216041908501,1,3,1,3,22170502.0,22170696.0
9,2025-04-01 02:00:00.000 UTC,0x9d39a5de30e57443bff2a8307a4256c8797a3497,sUSDe,NaN,2022925936809196800000000,-2022925936809196800000000,0,1,0,1,NaN,22170862.0


In [13]:
#7711171
# df_1_supply_withdraw = run.load("decimal_reference_part1")
# print(f"{TABLE}: {df_1_supply_withdraw.shape[0]} rows x {df_1_supply_withdraw.shape[1]} cols")
# display(df_1_supply_withdraw.dtypes.to_frame("dtype"))
# display(df_1_supply_withdraw.head(PREVIEW_ROWS))
df_0_supply_withdraw.drop(["net_supply_flow_raw", "latest_supply_block", "latest_withdraw_block"], axis=1, inplace=True)
# df_1_supply_withdraw.drop(["metric", "unit"], axis=1, inplace=True)
# df_1_supply_withdraw = df_1_supply_withdraw.drop(df_1_supply_withdraw.index[[0, 1]])

#query_7798226 =  7714873
df_oracle_price_usd_eth = run.load("oracle_price_usd_eth_weth_6h")
sc.check_frame(df_oracle_price_usd_eth, PROCESS_SCHEMAS["oracle_price_usd_eth_weth_6h"],
               table="oracle_price_usd_eth_weth_6h", stage="transform:input")
print(f"{TABLE}: {df_oracle_price_usd_eth.shape[0]} rows x {df_oracle_price_usd_eth.shape[1]} cols")
display(df_oracle_price_usd_eth.dtypes.to_frame("dtype"))
df_oracle_price_usd_eth


supply_withdraw: 206330 rows x 7 cols


,dtype
time_bucket,object
asset,object
symbol,object
decimals,int64
avg_price_usd,float64
avg_price_eth,float64
price_points,int64


,time_bucket,asset,symbol,decimals,avg_price_usd,avg_price_eth,price_points
0,2025-04-01 02:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,18,0.187991,0.000103,2
1,2025-04-01 02:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,18,82427.632083,45.116954,2
2,2025-04-01 02:00:00,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,6,1.082562,0.000593,2
3,2025-04-01 02:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,18,5.995833,0.003282,2
4,2025-04-01 02:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,8,82554.128333,45.186191,2
...,...,...,...,...,...,...,...
206325,2026-03-31 00:00:00,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,18,1.000157,0.000493,2
206326,2026-03-31 00:00:00,0xdefa4e8a7bcba345f687a2f1456f5edd9ce97202,KNC,18,0.148405,0.000073,2
206327,2026-03-31 00:00:00,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,6,1.000128,0.000493,2
206328,2026-03-31 00:00:00,0xf1c9acdc66974dfb6decb12aa385b9cd01190e38,osETH,18,2155.283333,1.062244,2


In [14]:

stats = adv.statistical_validation(df_oracle_price_usd_eth, columns=["avg_price_usd", "avg_price_eth"],save=False)
print(stats[["column", "n_null", "null_pct", "n_zero", "zero_pct"]])

          column  n_null  null_pct  n_zero  zero_pct
0  avg_price_usd       0       0.0       0       0.0
1  avg_price_eth       0       0.0       0       0.0


In [15]:
# Cell — divide raw amount columns by 10**decimals -> real token units
# decimals come from the reference table (df_1); raw_token_amount rows only.
df_0_supply_withdraw_scaled = tf.scale_by_decimals(df_0_supply_withdraw, df_oracle_price_usd_eth, drop_raw=True)

print(f"raw cols scaled: {tf.raw_amount_columns(df_0_supply_withdraw)}")
display(df_0_supply_withdraw_scaled.head(PREVIEW_ROWS))
# Cell — validation checks on the scaled amount columns of df_0 (after scale_by_decimals)
import adv_validation as av

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
amount_cols = [c[: -len(tf.RAW_SUFFIX)] for c in tf.raw_amount_columns(df_0_supply_withdraw)]
print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)
for col in amount_cols:
    print(f"\n===== {col} =====")
    neg = av.negative_value_check(df_0_supply_withdraw_scaled, col, plot=False)   # any negatives? how many?
    # rng = av.range_check(df_0_supply_withdraw_scaled, col, plot=False)            # observed value range
    # dev = av.deviation_score(df_0_supply_withdraw_scaled, col, plot=False)        # per-row score in [-1, 1]

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")

raw cols scaled: ['supply_amount_raw', 'withdrawal_amount_raw']


,time_bucket,asset,asset_symbol,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount,withdrawal_amount
0,2025-04-01 02:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,4,1,3,1,80.937906,3.352482e+00
1,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,12,2,5,1,7.204718,2.464649e+00
2,2025-04-01 02:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,1,1,1,1,20.000000,2.557563e+02
3,2025-04-01 02:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,2,0,2,0,101.772080,NaN
4,2025-04-01 02:00:00.000 UTC,0x5f98805a4e8be255a32880fdec7f6728c6568ba0,LUSD,1,1,1,1,18.242549,1.009207e+02
5,2025-04-01 02:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,0,1,0,1,NaN,3.901602e+05
6,2025-04-01 02:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,1,5,1,1,584566.561797,5.431237e+05
7,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,3,1,2,1,22.598483,2.695115e+01
8,2025-04-01 02:00:00.000 UTC,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,AAVE,1,3,1,3,0.083734,1.923276e+00
9,2025-04-01 02:00:00.000 UTC,0x9d39a5de30e57443bff2a8307a4256c8797a3497,sUSDe,0,1,0,1,NaN,2.022926e+06


amount columns: ['supply_amount', 'withdrawal_amount']

===== supply_amount =====
negatives : 0 of 66824 (0.0%)

===== withdrawal_amount =====
negatives : 0 of 62525 (0.0%)


In [16]:
amount_cols = [
    "supply_amount",
    "withdrawal_amount",
]
df_3_supply_withdraw = tf.multiply_by_price(df_0_supply_withdraw_scaled, df_oracle_price_usd_eth, amount_cols)
df_3_supply_withdraw

,time_bucket,asset,asset_symbol,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount,withdrawal_amount,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth
0,2025-04-01 02:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,4,1,3,1,8.093791e+01,3.352482e+00,485.290192,0.265625,20.100921,0.011002
1,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,12,2,5,1,7.204718e+00,2.464649e+00,594779.182181,325.553748,203466.945715,111.368099
2,2025-04-01 02:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,1,1,1,1,2.000000e+01,2.557563e+02,20.012268,0.010954,255.913146,0.140075
3,2025-04-01 02:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,2,0,2,0,1.017721e+02,NaN,1381.047128,0.755919,NaN,NaN
4,2025-04-01 02:00:00.000 UTC,0x5f98805a4e8be255a32880fdec7f6728c6568ba0,LUSD,1,1,1,1,1.824255e+01,1.009207e+02,18.275582,0.010003,101.103430,0.055339
...,...,...,...,...,...,...,...,...,...,...,...,...,...
84604,2026-03-31 22:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,USDtb,0,1,0,1,NaN,5.885586e+02,NaN,NaN,NaN,NaN
84605,2026-03-31 22:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,6,5,5,5,4.748101e+00,2.990320e+00,NaN,NaN,NaN,NaN
84606,2026-03-31 22:00:00.000 UTC,0xd11c452fc99cf405034ee446803b6f6c1f6d5ed8,tETH,0,1,0,1,NaN,2.131000e-02,NaN,NaN,NaN,NaN
84607,2026-03-31 22:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,29,23,17,13,2.363151e+07,3.537996e+06,NaN,NaN,NaN,NaN


In [17]:
amount_cols = [
    "supply_tx_count",
    "withdrawal_tx_count",
    "unique_suppliers",
    "unique_withdraw_users",
    "supply_amount_value_usd",
    "supply_amount_value_eth",
    "withdrawal_amount_value_usd",
    "withdrawal_amount_value_eth",
]

df_4_supply_withdraw = tf.aggregate_by_time_bucket(df_3_supply_withdraw, "time_bucket", amount_cols, agg_func='sum')
df_4_supply_withdraw

,time_bucket,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth
0,2025-04-01 02:00:00.000 UTC,107,112,54,60,1.814934e+08,99340.849932,9.766776e+07,53458.667287
1,2025-04-01 04:00:00.000 UTC,116,105,60,55,2.512122e+08,136862.261136,2.665668e+08,145227.537741
2,2025-04-01 06:00:00.000 UTC,72,100,38,54,1.096440e+08,59557.933326,1.329771e+08,72232.382778
3,2025-04-01 08:00:00.000 UTC,129,115,58,48,6.726555e+08,362808.719631,6.691233e+08,360903.546607
4,2025-04-01 10:00:00.000 UTC,138,118,68,53,3.680422e+08,195833.482106,3.695117e+08,196615.376327
...,...,...,...,...,...,...,...,...,...
4374,2026-03-31 14:00:00.000 UTC,187,195,93,104,0.000000e+00,0.000000,0.000000e+00,0.000000
4375,2026-03-31 16:00:00.000 UTC,228,157,121,104,0.000000e+00,0.000000,0.000000e+00,0.000000
4376,2026-03-31 18:00:00.000 UTC,223,193,94,123,0.000000e+00,0.000000,0.000000e+00,0.000000
4377,2026-03-31 20:00:00.000 UTC,164,155,102,99,0.000000e+00,0.000000,0.000000e+00,0.000000


In [18]:

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
amount_cols = [
    "supply_tx_count",
    "withdrawal_tx_count",
    "unique_suppliers",
    "unique_withdraw_users",
    "supply_amount_value_usd",
    "supply_amount_value_eth",
    "withdrawal_amount_value_usd",
    "withdrawal_amount_value_eth",
]
print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)
for col in amount_cols:
    print(f"\n===== {col} =====")


    neg = adv.negative_value_check(df_4_supply_withdraw, col, plot=False)
    # rng = av.range_check(df_4_supply_withdraw, col, plot=False)
    # dev = av.deviation_score(df_4_supply_withdraw, col, plot=False)

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")


amount columns: ['supply_tx_count', 'withdrawal_tx_count', 'unique_suppliers', 'unique_withdraw_users', 'supply_amount_value_usd', 'supply_amount_value_eth', 'withdrawal_amount_value_usd', 'withdrawal_amount_value_eth']

===== supply_tx_count =====
negatives : 0 of 4379 (0.0%)

===== withdrawal_tx_count =====
negatives : 0 of 4379 (0.0%)

===== unique_suppliers =====
negatives : 0 of 4379 (0.0%)

===== unique_withdraw_users =====
negatives : 0 of 4379 (0.0%)

===== supply_amount_value_usd =====
negatives : 0 of 4379 (0.0%)

===== supply_amount_value_eth =====
negatives : 0 of 4379 (0.0%)

===== withdrawal_amount_value_usd =====
negatives : 0 of 4379 (0.0%)

===== withdrawal_amount_value_eth =====
negatives : 0 of 4379 (0.0%)


In [19]:
stats = adv.statistical_validation(df_4_supply_withdraw, amount_cols, save=False)

print(stats[["column", 
             "null_pct", "n_zero", "zero_pct", "negative_pct",
    "n_sentinel", "mean", "std", "cv",
    # "min", "p01", "p05", "q25", "median", "q75", 
    # "p95", "p99", #"max",
    # "iqr", "mad", "skewness", "excess_kurtosis",
    # "n_outliers_iqr", "outlier_iqr_pct", "n_outliers_mad", "outlier_mad_pct",
    "heavy_tailed", "success",]])

                        column  null_pct  n_zero  zero_pct  negative_pct  \
0              supply_tx_count       0.0       0    0.0000           0.0   
1          withdrawal_tx_count       0.0       0    0.0000           0.0   
2             unique_suppliers       0.0       0    0.0000           0.0   
3        unique_withdraw_users       0.0       0    0.0000           0.0   
4      supply_amount_value_usd       0.0      11    0.2512           0.0   
5      supply_amount_value_eth       0.0      11    0.2512           0.0   
6  withdrawal_amount_value_usd       0.0      11    0.2512           0.0   
7  withdrawal_amount_value_eth       0.0      11    0.2512           0.0   

   n_sentinel          mean           std        cv  heavy_tailed  success  
0           0  1.851528e+02  9.859774e+01  0.532521          True     True  
1           0  1.557817e+02  8.968599e+01  0.575716          True     True  
2           0  9.767687e+01  4.647282e+01  0.475781          True     True  
3      

In [20]:
# load one table (read-only) and display it
# THE VALUES ARE NOT RAW, IT WAS PRE SCALED BY SQL
TABLE = "borrow_repay"  # <-- a label from run.tables()

df_0_borrow = run.load(TABLE)
sc.check_frame(df_0_borrow, PROCESS_SCHEMAS[TABLE], table=TABLE, stage="transform:input")
print(f"{TABLE}: {df_0_borrow.shape[0]} rows x {df_0_borrow.shape[1]} cols")
display(df_0_borrow.dtypes.to_frame("dtype"))
display(df_0_borrow.head(PREVIEW_ROWS))

borrow_repay: 54198 rows x 15 cols


,dtype
time_bucket,object
asset,object
asset_symbol,object
borrow_amount,float64
repay_amount,float64
net_debt_flow,float64
borrow_tx_count,int64
repay_tx_count,int64
stable_borrow_tx_count,int64
variable_borrow_tx_count,int64


,time_bucket,asset,asset_symbol,borrow_amount,repay_amount,net_debt_flow,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,last_borrow_rate,latest_borrow_block,latest_repay_block
0,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,4.000000e-01,1.060043e+00,-6.600429e-01,2,4,0,2,1,3,5102958609642485824368628,22170766.0,22170811.0
1,2025-04-01 02:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,4.044077e+04,1.129596e+03,3.931117e+04,3,2,0,3,3,2,45000000000000000000000000,22170868.0,22170845.0
2,2025-04-01 02:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,4.200000e+02,2.000000e+01,4.000000e+02,2,1,0,2,2,1,33221773480495132622398080,22170778.0,22170797.0
3,2025-04-01 02:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,1.500000e+05,NaN,1.500000e+05,1,0,0,1,1,0,48094819243714186856515190,22170769.0,NaN
4,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,2.000000e+02,NaN,2.000000e+02,1,0,0,1,1,0,4967538120472516497064990,22170544.0,NaN
5,2025-04-01 02:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,8.277720e+06,1.948000e+07,-1.120228e+07,16,17,0,16,11,10,46030830777063318637077796,22170914.0,22170930.0
6,2025-04-01 02:00:00.000 UTC,0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f,SNX,1.789983e+04,4.820925e+03,1.307890e+04,1,2,0,1,1,2,89416998540159967524840963,22170789.0,22170928.0
7,2025-04-01 02:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,6.983152e+02,5.939031e+02,1.044121e+02,14,13,0,14,3,4,26734561820823454479814960,22170932.0,22170930.0
8,2025-04-01 02:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,9.109636e+04,7.834299e+06,-7.743202e+06,14,7,0,14,14,4,48260450698177106607739218,22170925.0,22170917.0
9,2025-04-01 02:00:00.000 UTC,0xf939e0a03fb07f59a73314e73794be0e57ac1b4e,crvUSD,NaN,5.020625e+04,-5.020625e+04,0,1,0,0,0,1,NaN,NaN,22170524.0


In [21]:
# df_1 = run.load("decimal_reference_part2")
# print(f"{TABLE}: {df_1.shape[0]} rows x {df_1.shape[1]} cols")
# display(df_1.dtypes.to_frame("dtype"))
# display(df_1.head(PREVIEW_ROWS))

df_0_borrow["last_borrow_rate"] = df_0_borrow["last_borrow_rate"].astype(float) / 10**27
df_0_borrow

,time_bucket,asset,asset_symbol,borrow_amount,repay_amount,net_debt_flow,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,last_borrow_rate,latest_borrow_block,latest_repay_block
0,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,4.000000e-01,1.060043,-6.600429e-01,2,4,0,2,1,3,0.005103,22170766.0,22170811.0
1,2025-04-01 02:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,4.044077e+04,1129.596248,3.931117e+04,3,2,0,3,3,2,0.045,22170868.0,22170845.0
2,2025-04-01 02:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,4.200000e+02,20.000005,4.000000e+02,2,1,0,2,2,1,0.033222,22170778.0,22170797.0
3,2025-04-01 02:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,1.500000e+05,NaN,1.500000e+05,1,0,0,1,1,0,0.048095,22170769.0,NaN
4,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,2.000000e+02,NaN,2.000000e+02,1,0,0,1,1,0,0.004968,22170544.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54193,2026-03-31 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,4.184436e+06,542927.995209,3.641508e+06,24,18,0,24,22,15,0.034642,24773806.0,24773857.0
54194,2026-03-31 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,1.079670e+02,249.990952,-1.420239e+02,7,4,0,7,4,4,0.022331,24773853.0,24773817.0
54195,2026-03-31 00:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,USDtb,1.500000e+02,NaN,1.500000e+02,2,0,0,2,2,0,0.02225,24773674.0,NaN
54196,2026-03-31 00:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,5.312990e-03,0.005313,-1.000000e-08,1,1,0,1,1,1,0.003051,24773579.0,24773579.0


In [22]:
#oracle_price_usd_eth_weth_6h =  7714873
df_oracle_price_usd_eth 

,time_bucket,asset,symbol,decimals,avg_price_usd,avg_price_eth,price_points
0,2025-04-01 02:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,18,0.187991,0.000103,2
1,2025-04-01 02:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,18,82427.632083,45.116954,2
2,2025-04-01 02:00:00,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,6,1.082562,0.000593,2
3,2025-04-01 02:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,18,5.995833,0.003282,2
4,2025-04-01 02:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,8,82554.128333,45.186191,2
...,...,...,...,...,...,...,...
206325,2026-03-31 00:00:00,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,18,1.000157,0.000493,2
206326,2026-03-31 00:00:00,0xdefa4e8a7bcba345f687a2f1456f5edd9ce97202,KNC,18,0.148405,0.000073,2
206327,2026-03-31 00:00:00,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,6,1.000128,0.000493,2
206328,2026-03-31 00:00:00,0xf1c9acdc66974dfb6decb12aa385b9cd01190e38,osETH,18,2155.283333,1.062244,2


In [23]:
df_0_borrow.drop(["net_debt_flow","latest_borrow_block", "latest_repay_block"], axis=1, inplace=True)
DF_common = df_0_borrow[["time_bucket", "asset", "asset_symbol", "last_borrow_rate"]].copy()
df_0_borrow.drop(["last_borrow_rate"], axis=1, inplace=True)
df_0_borrow

,time_bucket,asset,asset_symbol,borrow_amount,repay_amount,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers
0,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,4.000000e-01,1.060043,2,4,0,2,1,3
1,2025-04-01 02:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,4.044077e+04,1129.596248,3,2,0,3,3,2
2,2025-04-01 02:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,4.200000e+02,20.000005,2,1,0,2,2,1
3,2025-04-01 02:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,1.500000e+05,NaN,1,0,0,1,1,0
4,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,2.000000e+02,NaN,1,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...
54193,2026-03-31 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,4.184436e+06,542927.995209,24,18,0,24,22,15
54194,2026-03-31 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,1.079670e+02,249.990952,7,4,0,7,4,4
54195,2026-03-31 00:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,USDtb,1.500000e+02,NaN,2,0,0,2,2,0
54196,2026-03-31 00:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,5.312990e-03,0.005313,1,1,0,1,1,1


In [24]:
# Cell — divide borrow_repay raw amount columns by 10**decimals -> real token units
# decimals come from df_1_borrow (the oracle_price table's `decimals` column), per asset.
# df_2_borrow = tf.scale_by_decimals(df_0_borrow, df_1_borrow)

# print(f"raw cols scaled: {tf.raw_amount_columns(df_0_borrow)}")
# display(df_2_borrow.head(PREVIEW_ROWS))

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
# amount_cols = [c[: -len(tf.RAW_SUFFIX)] for c in tf.raw_amount_columns(df_0_borrow)]
# print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)

amount_cols = [
    "borrow_amount",
    "repay_amount",
]
for col in amount_cols:
    print(f"\n===== {col} =====")
    neg = av.negative_value_check(df_0_borrow, col, plot=False)   # any negatives? how many?
    # rng = av.range_check(df_2_borrow, col, plot=False)            # observed value range
    # dev = av.deviation_score(df_2_borrow, col, plot=False)        # per-row score in [-1, 1]

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")




===== borrow_amount =====
negatives : 0 of 45301 (0.0%)

===== repay_amount =====
negatives : 0 of 44817 (0.0%)


In [25]:


df_2_borrow = tf.multiply_by_price(df_0_borrow, df_oracle_price_usd_eth, amount_cols)
df_2_borrow



,time_bucket,asset,asset_symbol,borrow_amount,repay_amount,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,borrow_amount_value_usd,borrow_amount_value_eth,repay_amount_value_usd,repay_amount_value_eth
0,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,4.000000e-01,1.060043,2,4,0,2,1,3,3.302165e+04,18.074477,87510.915129,47.899300
1,2025-04-01 02:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,4.044077e+04,1129.596248,3,2,0,3,3,2,4.045823e+04,22.144902,1130.083857,0.618554
2,2025-04-01 02:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,4.200000e+02,20.000005,2,1,0,2,2,1,4.202576e+02,0.230029,20.012273,0.010954
3,2025-04-01 02:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,1.500000e+05,NaN,1,0,0,1,1,0,1.500965e+05,82.155660,NaN,NaN
4,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,2.000000e+02,NaN,1,0,0,1,1,0,4.376308e+05,239.538168,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54193,2026-03-31 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,4.184436e+06,542927.995209,24,18,0,24,22,15,4.184672e+06,2062.457100,542958.580153,267.602510
54194,2026-03-31 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,1.079670e+02,249.990952,7,4,0,7,4,4,2.190643e+05,107.967016,507229.767282,249.990952
54195,2026-03-31 00:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,USDtb,1.500000e+02,NaN,2,0,0,2,2,0,1.499275e+02,0.073893,NaN,NaN
54196,2026-03-31 00:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,5.312990e-03,0.005313,1,1,0,1,1,1,3.543657e+02,0.174652,354.366381,0.174652


In [26]:
borrow_agg_cols = [
    "borrow_tx_count",
    "repay_tx_count",
    # "stable_borrow_tx_count",
    "variable_borrow_tx_count",
    "unique_borrowers",
    "unique_repayers",
    "borrow_amount_value_usd",
    "borrow_amount_value_eth",
    "repay_amount_value_usd",
    "repay_amount_value_eth",
]

df_3_borrow = tf.aggregate_by_time_bucket(df_2_borrow, "time_bucket", borrow_agg_cols, agg_func='sum')
# df_3_borrow.drop(["stable_borrow_tx_count"], axis=1, inplace=True )

df_3_borrow

,time_bucket,borrow_tx_count,repay_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,borrow_amount_value_usd,borrow_amount_value_eth,repay_amount_value_usd,repay_amount_value_eth
0,2025-04-01 02:00:00.000 UTC,54,47,54,37,27,1.032549e+07,5651.683080,2.855941e+07,15632.055680
1,2025-04-01 04:00:00.000 UTC,58,58,58,47,42,1.360886e+07,7414.203082,2.294215e+07,12499.053578
2,2025-04-01 06:00:00.000 UTC,48,45,48,37,38,9.805061e+06,5326.065746,1.644968e+07,8935.388910
3,2025-04-01 08:00:00.000 UTC,64,67,64,43,36,2.930883e+07,15808.263478,5.574171e+07,30065.281708
4,2025-04-01 10:00:00.000 UTC,95,71,95,67,42,1.601311e+07,8520.523856,3.426404e+07,18231.785356
...,...,...,...,...,...,...,...,...,...,...
4363,2026-03-30 16:00:00.000 UTC,112,101,112,86,68,5.067923e+07,24561.015042,2.762900e+07,13390.077453
4364,2026-03-30 18:00:00.000 UTC,113,97,113,83,62,1.125271e+07,5467.748299,1.729953e+07,8405.886893
4365,2026-03-30 20:00:00.000 UTC,104,92,104,74,72,9.916947e+06,4888.783277,9.870147e+06,4865.700859
4366,2026-03-30 22:00:00.000 UTC,75,80,75,61,57,4.707848e+07,23198.456236,3.544758e+07,17467.176575


In [27]:
stats_borr = adv.statistical_validation(df_3_borrow, borrow_agg_cols, save=False)

# features -> rows, metrics -> columns
metric_cols = ["mean", "median", "std", "cv", "p95", "p99"]
stats = stats_borr.set_index("column")[metric_cols]

print(stats)

                                  mean        median           std        cv  \
column                                                                         
borrow_tx_count           1.056635e+02  9.500000e+01  5.206797e+01  0.492772   
repay_tx_count            8.886653e+01  7.200000e+01  7.180547e+01  0.808015   
variable_borrow_tx_count  1.056635e+02  9.500000e+01  5.206797e+01  0.492772   
unique_borrowers          7.298832e+01  6.700000e+01  3.042017e+01  0.416781   
unique_repayers           5.416071e+01  4.500000e+01  4.478458e+01  0.826883   
borrow_amount_value_usd   4.651291e+07  3.031552e+07  5.843244e+07  1.256263   
borrow_amount_value_eth   1.578487e+04  1.055787e+04  1.866181e+04  1.182259   
repay_amount_value_usd    4.565727e+07  2.850048e+07  5.848805e+07  1.281024   
repay_amount_value_eth    1.554396e+04  1.000056e+04  1.894298e+04  1.218671   

                                   p95           p99  
column                                                
borrow_tx

In [28]:
TABLE = "reserve_state_rates"  

df_0_reserve_state_rates = run.load(TABLE)
sc.check_frame(df_0_reserve_state_rates, PROCESS_SCHEMAS[TABLE], table=TABLE, stage="transform:input")
df_0_reserve_state_rates.rename(columns={"update_count": "update_count_liquidity"}, inplace=True)
print(f"{TABLE}: {df_0_reserve_state_rates.shape[0]} rows x {df_0_reserve_state_rates.shape[1]} cols")
display(df_0_reserve_state_rates.dtypes.to_frame("dtype"))
display(df_0_reserve_state_rates.head(PREVIEW_ROWS))

reserve_state_rates: 99143 rows x 8 cols


,dtype
time_bucket,object
asset,object
liquidity_rate,float64
variable_borrow_rate,float64
stable_borrow_rate,float64
liquidity_index,float64
variable_borrow_index,float64
update_count_liquidity,int64


,time_bucket,asset,liquidity_rate,variable_borrow_rate,stable_borrow_rate,liquidity_index,variable_borrow_index,update_count_liquidity
0,2025-04-01 02:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,0.000240,0.006838,0.0,1.001884,1.016139,5
1,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,0.000260,0.005101,0.0,1.003301,1.022170,21
2,2025-04-01 02:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,0.000000,0.045000,0.0,1.000000,1.123769,5
3,2025-04-01 02:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,0.011037,0.033222,0.0,1.038794,1.074808,5
4,2025-04-01 02:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,0.000246,0.006911,0.0,1.000689,1.007202,2
5,2025-04-01 02:00:00.000 UTC,0x5f98805a4e8be255a32880fdec7f6728c6568ba0,0.070883,0.108151,0.0,1.103381,1.165427,2
6,2025-04-01 02:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,0.026601,0.048095,0.0,1.121824,1.179228,2
7,2025-04-01 02:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,0.011034,0.044527,0.0,1.053432,1.095327,6
8,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,0.001172,0.004968,0.0,1.001093,1.009286,6
9,2025-04-01 02:00:00.000 UTC,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,0.000000,0.000000,0.0,1.000000,1.000000,4


In [29]:
reserve_cols = [
    "liquidity_rate",
    "stable_borrow_rate",
    "variable_borrow_rate",
    "liquidity_index",
    "variable_borrow_index",
]

# for col in reserve_cols:
# ALREADY DONE BY SQL
#     df_0_reserve_state_rates[col] = df_0_reserve_state_rates[col].astype(float) / 10**27

DF_common_1 = DF_common.merge(
    df_0_reserve_state_rates,
      on=["time_bucket", "asset"], how="left", suffixes=("_x", "_y"))

DF_common_1.drop(["stable_borrow_rate"], axis=1, inplace=True)
DF_common_1

,time_bucket,asset,asset_symbol,last_borrow_rate,liquidity_rate,variable_borrow_rate,liquidity_index,variable_borrow_index,update_count_liquidity
0,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,0.005103,0.000260,0.005101,1.003301,1.022170,21
1,2025-04-01 02:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0.045,0.000000,0.045000,1.000000,1.123769,5
2,2025-04-01 02:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,0.033222,0.011037,0.033222,1.038794,1.074808,5
3,2025-04-01 02:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,0.048095,0.026601,0.048095,1.121824,1.179228,2
4,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,0.004968,0.001172,0.004968,1.001093,1.009286,6
...,...,...,...,...,...,...,...,...,...
54193,2026-03-31 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,0.034642,0.024840,0.034642,1.165281,1.222860,114
54194,2026-03-31 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,0.022331,0.016594,0.022331,1.061964,1.094464,72
54195,2026-03-31 00:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,USDtb,0.02225,0.007921,0.022250,1.013739,1.031589,2
54196,2026-03-31 00:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,0.003051,0.000017,0.003051,1.000195,1.004011,6


In [30]:
common_cols = [
    "last_borrow_rate",
    "liquidity_rate",
    "variable_borrow_rate",
    "liquidity_index",
    "variable_borrow_index",
]

stats = pd.DataFrame({
    "mean": DF_common_1[common_cols].mean(),
    "median": DF_common_1[common_cols].median(),
    "std": DF_common_1[common_cols].std(),
    "variance": DF_common_1[common_cols].var(),
    "cv": DF_common_1[common_cols].std() / DF_common_1[common_cols].mean(),
    "p95": DF_common_1[common_cols].quantile(0.95),
    "p99": DF_common_1[common_cols].quantile(0.99)
})

print(stats)

                           mean    median       std  variance        cv  \
last_borrow_rate        0.03505  0.036207  0.038025  0.001446  1.084895   
liquidity_rate         0.015778  0.012786  0.019023  0.000362   1.20564   
variable_borrow_rate     0.0333   0.03545  0.027096  0.000734  0.813693   
liquidity_index         1.04932  1.019381  0.057466  0.003302  0.054765   
variable_borrow_index  1.094467  1.080621  0.080826  0.006533   0.07385   

                            p95       p99  
last_borrow_rate       0.062958  0.106419  
liquidity_rate         0.042809  0.054291  
variable_borrow_rate   0.063259  0.097939  
liquidity_index        1.154338  1.164819  
variable_borrow_index  1.222507  1.252292  


In [31]:
TABLE = "reserve_config"

df_0_reserve_config = run.load(TABLE)
sc.check_frame(df_0_reserve_config, PROCESS_SCHEMAS[TABLE], table=TABLE, stage="transform:input")
print(f"{TABLE}: {df_0_reserve_config.shape[0]} rows x {df_0_reserve_config.shape[1]} cols")
display(df_0_reserve_config.dtypes.to_frame("dtype"))
display(df_0_reserve_config.head(PREVIEW_ROWS))
summary = pd.DataFrame({
    "null_count": df_0_reserve_config.isna().sum(),
    "null_pct": df_0_reserve_config.isna().mean() * 100,
})
print(summary)

# DECISION (ERR-13): reserve_config is DROPPED from the feature scope — the extract is
# inconsistent, and the old valuation cells double-scaled debt_ceiling and priced a
# USD-denominated ceiling by asset price. Load/inspect checkpoint only, nothing reads it.


reserve_config: 232175 rows x 11 cols


,dtype
time_bucket,object
asset,object
asset_symbol,object
supply_cap,int64
borrow_cap,int64
debt_ceiling,float64
reserve_factor,float64
liquidation_threshold,float64
liquidation_bonus,float64
ltv,float64


,time_bucket,asset,asset_symbol,supply_cap,borrow_cap,debt_ceiling,reserve_factor,liquidation_threshold,liquidation_bonus,ltv,config_event_count
0,2025-04-01 02:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,9000000,475200,4500000.0,0.20,0.67,1.075,0.57,0
1,2025-04-01 02:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,2200,275,0.0,0.50,0.78,1.075,0.73,0
2,2025-04-01 02:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,6000000,330000,17000000.0,0.20,0.74,1.100,0.65,0
3,2025-04-01 02:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,47500,28000,0.0,0.50,0.78,1.050,0.73,0
4,2025-04-01 02:00:00,0x3432b6a60d23ca0dfca7761b7ab56459d9c964d0,FXS,1200000,330000,4000000.0,0.20,0.42,1.100,0.00,0
5,2025-04-01 02:00:00,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0,192500000,0.0,0.00,0.00,0.000,0.00,0
6,2025-04-01 02:00:00,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,120000000,110000000,50000000.0,0.25,0.75,1.085,0.72,0
7,2025-04-01 02:00:00,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,20000000,13000000,0.0,0.20,0.71,1.070,0.66,0
8,2025-04-01 02:00:00,0x5a98fcbea516cf06857215779fd812ca3bef1b32,LDO,8000000,1500000,7500000.0,0.20,0.50,1.090,0.40,0
9,2025-04-01 02:00:00,0x5f98805a4e8be255a32880fdec7f6728c6568ba0,LUSD,7000000,5000000,0.0,0.20,0.77,1.045,0.00,0


                       null_count  null_pct
time_bucket                     0       0.0
asset                           0       0.0
asset_symbol                    0       0.0
supply_cap                      0       0.0
borrow_cap                      0       0.0
debt_ceiling                    0       0.0
reserve_factor                  0       0.0
liquidation_threshold           0       0.0
liquidation_bonus               0       0.0
ltv                             0       0.0
config_event_count              0       0.0


In [32]:
df_oracle_price_usd_eth

,time_bucket,asset,symbol,decimals,avg_price_usd,avg_price_eth,price_points
0,2025-04-01 02:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,18,0.187991,0.000103,2
1,2025-04-01 02:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,18,82427.632083,45.116954,2
2,2025-04-01 02:00:00,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,6,1.082562,0.000593,2
3,2025-04-01 02:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,18,5.995833,0.003282,2
4,2025-04-01 02:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,8,82554.128333,45.186191,2
...,...,...,...,...,...,...,...
206325,2026-03-31 00:00:00,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,18,1.000157,0.000493,2
206326,2026-03-31 00:00:00,0xdefa4e8a7bcba345f687a2f1456f5edd9ce97202,KNC,18,0.148405,0.000073,2
206327,2026-03-31 00:00:00,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,6,1.000128,0.000493,2
206328,2026-03-31 00:00:00,0xf1c9acdc66974dfb6decb12aa385b9cd01190e38,osETH,18,2155.283333,1.062244,2


In [33]:
TABLE = "liquidation"

df_0_liq = run.load(TABLE)
sc.check_frame(df_0_liq, PROCESS_SCHEMAS[TABLE], table=TABLE, stage="transform:input")
print(f"{TABLE}: {df_0_liq.shape[0]} rows x {df_0_liq.shape[1]} cols")
display(df_0_liq.dtypes.to_frame("dtype"))
# display(df_0_liq.head(PREVIEW_ROWS))
df_0_liq

liquidation: 4468 rows x 12 cols


,dtype
time_bucket,object
asset,object
asset_symbol,object
liquidated_collateral_raw,object
liquidation_debt_covered_raw,object
as_collateral_tx_count,int64
as_debt_tx_count,int64
liquidation_tx_count,int64
receive_atoken_count,int64
unique_liquidated_users,int64


,time_bucket,asset,asset_symbol,liquidated_collateral_raw,liquidation_debt_covered_raw,as_collateral_tx_count,as_debt_tx_count,liquidation_tx_count,receive_atoken_count,unique_liquidated_users,unique_liquidators,latest_liquidation_block
0,2025-04-01 08:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,17432984242,0,1,0,1,0,1,1,22172329
1,2025-04-01 08:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,0,10941402645440751977,0,2,2,0,2,2,22172329
2,2025-04-01 08:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,4541455,0,1,0,1,0,1,1,22172327
3,2025-04-01 10:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,0,2836667175746209741782,0,1,1,0,1,1,22172888
4,2025-04-01 10:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,41214339648,0,1,0,1,0,1,1,22172888
...,...,...,...,...,...,...,...,...,...,...,...,...
4463,2026-03-30 20:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,0,1130933496,0,1,1,0,1,1,24772182
4464,2026-03-31 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,16278377,0,1,0,1,0,1,1,24773651
4465,2026-03-31 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,6337065898668302564,0,1,0,1,0,1,1,24773579
4466,2026-03-31 00:00:00.000 UTC,0x8292bb45bf1ee4d140127049757c2e0ff06317ed,RLUSD,0,10332283482842769560042,0,1,1,0,1,1,24773651


In [34]:
TABLE = "decimal_reference_part4"

df_1_liq = run.load(TABLE)
sc.check_frame(df_1_liq, PROCESS_SCHEMAS[TABLE], table=TABLE, stage="transform:input")
print(f"{TABLE}: {df_1_liq.shape[0]} rows x {df_1_liq.shape[1]} cols")
display(df_1_liq.dtypes.to_frame("dtype"))
display(df_1_liq.head(PREVIEW_ROWS))

decimal_reference_part4: 82 rows x 5 cols


,dtype
asset,object
asset_symbol,object
metric,object
decimals,int64
unit,object


,asset,asset_symbol,metric,decimals,unit
0,NaN,NaN,borrow_cap,0,whole_tokens
1,NaN,NaN,config_event_count,0,count
2,NaN,NaN,debt_ceiling,2,usd_2dp
3,NaN,NaN,latest_liquidation_block,0,block_number
4,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,liquidated_collateral_raw,18,raw_token_amount
5,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,liquidated_collateral_raw,18,raw_token_amount
6,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,liquidated_collateral_raw,18,raw_token_amount
7,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,liquidated_collateral_raw,8,raw_token_amount
8,0x3b3fb9c57858ef816833dc91565efcd85d96f634,PT-sUSDE-31JUL2025,liquidated_collateral_raw,18,raw_token_amount
9,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,liquidated_collateral_raw,18,raw_token_amount


In [35]:
df_2_liq = tf.scale_by_decimals(df_0_liq, df_oracle_price_usd_eth, drop_raw=True)
df_3_liq = tf.multiply_by_price(df_2_liq, df_oracle_price_usd_eth, ["liquidated_collateral", "liquidation_debt_covered"])
df_3_liq


,time_bucket,asset,asset_symbol,as_collateral_tx_count,as_debt_tx_count,liquidation_tx_count,receive_atoken_count,unique_liquidated_users,unique_liquidators,latest_liquidation_block,liquidated_collateral,liquidation_debt_covered,liquidated_collateral_value_usd,liquidated_collateral_value_eth,liquidation_debt_covered_value_usd,liquidation_debt_covered_value_eth
0,2025-04-01 08:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,1,0,1,0,1,1,22172329,17432.984242,0.000000,17444.527057,9.409033,0.000000,0.000000
1,2025-04-01 08:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,0,2,2,0,2,2,22172329,0.000000,10.941403,0.000000,0.000000,20285.665952,10.941403
2,2025-04-01 08:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,1,0,1,0,1,1,22172327,0.045415,0.000000,3785.518840,2.041781,0.000000,0.000000
3,2025-04-01 10:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,0,1,1,0,1,1,22172888,0.000000,2836.667176,0.000000,0.000000,39643.605726,21.094034
4,2025-04-01 10:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,1,0,1,0,1,1,22172888,41214.339648,0.000000,41239.021886,21.943161,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4463,2026-03-30 20:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,0,1,1,0,1,1,24772182,0.000000,1130.933496,0.000000,0.000000,1130.882604,0.557495
4464,2026-03-31 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,1,0,1,0,1,1,24773651,0.162784,0.000000,10841.077584,5.343114,0.000000,0.000000
4465,2026-03-31 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,1,0,1,0,1,1,24773579,6.337066,0.000000,15822.185462,7.798051,0.000000,0.000000
4466,2026-03-31 00:00:00.000 UTC,0x8292bb45bf1ee4d140127049757c2e0ff06317ed,RLUSD,0,1,1,0,1,1,24773651,0.000000,10332.283483,0.000000,0.000000,10333.545313,5.092991


In [36]:
df_3_liq.drop(["latest_liquidation_block"], axis=1, inplace=True)
liq_agg_cols = [
    "as_collateral_tx_count",
    "as_debt_tx_count",
    "liquidation_tx_count",
    "unique_liquidated_users",
    "unique_liquidators",
    "liquidated_collateral_value_usd",
    "liquidated_collateral_value_eth",
    "liquidation_debt_covered_value_usd",
    "liquidation_debt_covered_value_eth",
]
df_4_liq = tf.aggregate_by_time_bucket(df_3_liq, "time_bucket", liq_agg_cols, agg_func='sum')
df_4_liq

,time_bucket,as_collateral_tx_count,as_debt_tx_count,liquidation_tx_count,unique_liquidated_users,unique_liquidators,liquidated_collateral_value_usd,liquidated_collateral_value_eth,liquidation_debt_covered_value_usd,liquidation_debt_covered_value_eth
0,2025-04-01 08:00:00.000 UTC,2,2,4,4,4,21230.045897,11.450814,20285.665952,10.941403
1,2025-04-01 10:00:00.000 UTC,1,1,2,2,2,41239.021886,21.943161,39643.605726,21.094034
2,2025-04-01 16:00:00.000 UTC,2,2,4,4,4,20970.444109,11.118519,19269.584636,10.218210
3,2025-04-01 22:00:00.000 UTC,1,1,2,2,2,17.627285,0.009220,16.855512,0.008816
4,2025-04-02 02:00:00.000 UTC,1,1,2,2,2,2716.800395,1.433326,2524.128530,1.331694
...,...,...,...,...,...,...,...,...,...,...
1217,2026-03-29 14:00:00.000 UTC,1,1,2,2,2,166.503252,0.083385,155.396755,0.077823
1218,2026-03-30 00:00:00.000 UTC,13,13,26,24,14,84467.224300,42.560463,80496.323743,40.559669
1219,2026-03-30 18:00:00.000 UTC,2,2,4,4,2,57565.177978,27.970869,54685.376602,26.571944
1220,2026-03-30 20:00:00.000 UTC,1,1,2,2,2,1178.444683,0.580937,1130.882604,0.557495


In [37]:
TABLE = "collateral_toggle"

df_0_collateral = run.load(TABLE)
sc.check_frame(df_0_collateral, PROCESS_SCHEMAS[TABLE], table=TABLE, stage="transform:input")
print(f"{TABLE}: {df_0_collateral.shape[0]} rows x {df_0_collateral.shape[1]} cols")
display(df_0_collateral.dtypes.to_frame("dtype"))
display(df_0_collateral.head(PREVIEW_ROWS))

collateral_toggle: 60436 rows x 8 cols


,dtype
time_bucket,object
asset,object
asset_symbol,float64
collateral_enabled_count,int64
collateral_disabled_count,int64
unique_collateral_enable_users,int64
unique_collateral_disable_users,int64
latest_collateral_toggle_block,int64


,time_bucket,asset,asset_symbol,collateral_enabled_count,collateral_disabled_count,unique_collateral_enable_users,unique_collateral_disable_users,latest_collateral_toggle_block
0,2025-04-01 02:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,NaN,1,1,1,1,22170819
1,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,NaN,9,4,8,3,22170740
2,2025-04-01 02:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,NaN,1,0,1,0,22170829
3,2025-04-01 02:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,NaN,0,1,0,1,22170539
4,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,NaN,2,2,2,2,22170805
5,2025-04-01 02:00:00.000 UTC,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,NaN,3,4,3,4,22170696
6,2025-04-01 02:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,NaN,18,18,12,13,22170880
7,2025-04-01 02:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,NaN,32,24,20,12,22170909
8,2025-04-01 02:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,NaN,0,2,0,2,22170828
9,2025-04-01 02:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,NaN,24,31,11,17,22170925


In [38]:
collateral_agg_cols = [
    "collateral_enabled_count",
    "collateral_disabled_count",
    "unique_collateral_enable_users",
    "unique_collateral_disable_users"
]

df_1_collateral = tf.aggregate_by_time_bucket(df_0_collateral, "time_bucket", collateral_agg_cols, agg_func='sum')
df_1_collateral

,time_bucket,collateral_enabled_count,collateral_disabled_count,unique_collateral_enable_users,unique_collateral_disable_users
0,2025-04-01 02:00:00.000 UTC,91,88,59,56
1,2025-04-01 04:00:00.000 UTC,86,64,61,41
2,2025-04-01 06:00:00.000 UTC,65,59,41,36
3,2025-04-01 08:00:00.000 UTC,92,72,58,39
4,2025-04-01 10:00:00.000 UTC,110,83,68,43
...,...,...,...,...,...
4363,2026-03-30 16:00:00.000 UTC,50,56,39,45
4364,2026-03-30 18:00:00.000 UTC,63,63,50,51
4365,2026-03-30 20:00:00.000 UTC,50,52,42,44
4366,2026-03-30 22:00:00.000 UTC,36,38,31,33


In [39]:
TABLE = "flashloan"

df_0_flashloan = run.load(TABLE)
sc.check_frame(df_0_flashloan, PROCESS_SCHEMAS[TABLE], table=TABLE, stage="transform:input")
print(f"{TABLE}: {df_0_flashloan.shape[0]} rows x {df_0_flashloan.shape[1]} cols")
display(df_0_flashloan.dtypes.to_frame("dtype"))
display(df_0_flashloan.head(PREVIEW_ROWS))

flashloan: 24888 rows x 11 cols


,dtype
time_bucket,object
asset,object
asset_symbol,object
flashloan_amount_raw,object
flashloan_premium_raw,object
flashloan_tx_count,int64
unique_flashloan_initiators,int64
no_open_debt_flashloan_tx_count,int64
stable_flashloan_tx_count,int64
variable_flashloan_tx_count,int64


,time_bucket,asset,asset_symbol,flashloan_amount_raw,flashloan_premium_raw,flashloan_tx_count,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,stable_flashloan_tx_count,variable_flashloan_tx_count,latest_flashloan_block
0,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,202304487,101152,1,1,1,0,0,22170544
1,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,26964624265336475000,13482312132668238,1,1,1,0,0,22170805
2,2025-04-01 02:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,239703808517,87236780,12,9,10,0,2,22170873
3,2025-04-01 02:00:00.000 UTC,0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f,SNX,17899829513104235132999,0,1,1,0,0,1,22170789
4,2025-04-01 02:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,189907915020145031208,12554862589381195,6,3,3,0,3,22170930
5,2025-04-01 02:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,128282362697,24976946,5,5,4,0,1,22170443
6,2025-04-01 04:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,7244943441,3622472,4,4,4,0,0,22171433
7,2025-04-01 04:00:00.000 UTC,0x83f20f44975d03b1b09e64809b757c47f942beea,sDAI,17652792225787096817343,0,1,1,1,0,0,22171301
8,2025-04-01 04:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,270225012614,4468111,7,5,5,0,2,22171321
9,2025-04-01 04:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,100768950617987207865,8320408451142263,4,4,3,0,1,22171400


In [40]:
TABLE = "decimal_reference_part5"

df_2_flashloan = run.load(TABLE)
sc.check_frame(df_2_flashloan, PROCESS_SCHEMAS[TABLE], table=TABLE, stage="transform:input")
print(f"{TABLE}: {df_2_flashloan.shape[0]} rows x {df_2_flashloan.shape[1]} cols")
# display(df_2_flashloan.dtypes.to_frame("dtype"))
# display(df_2_flashloan.head(PREVIEW_ROWS))

# split the reference frame by its `metric` column
df_2_flashloan_amount  = df_2_flashloan[df_2_flashloan["metric"] == "flashloan_amount_raw"].copy()
df_2_flashloan_premium = df_2_flashloan[df_2_flashloan["metric"] == "flashloan_premium_raw"].copy()

display(df_2_flashloan_amount.head(PREVIEW_ROWS))
display(df_2_flashloan_premium.head(PREVIEW_ROWS))

decimal_reference_part5: 95 rows x 5 cols


,asset,asset_symbol,metric,decimals,unit
0,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,flashloan_amount_raw,18,raw_token_amount
1,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,flashloan_amount_raw,18,raw_token_amount
2,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,flashloan_amount_raw,6,raw_token_amount
3,0x1f84a51296691320478c98b8d77f2bbd17d34350,PT-USDe-5FEB2026,flashloan_amount_raw,18,raw_token_amount
4,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,flashloan_amount_raw,18,raw_token_amount
5,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,flashloan_amount_raw,8,raw_token_amount
6,0x356b8d89c1e1239cbbb9de4815c39a1474d5ba7d,syrupUSDT,flashloan_amount_raw,6,raw_token_amount
7,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,flashloan_amount_raw,18,raw_token_amount
8,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,flashloan_amount_raw,18,raw_token_amount
9,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,flashloan_amount_raw,18,raw_token_amount


,asset,asset_symbol,metric,decimals,unit
43,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,flashloan_premium_raw,18,raw_token_amount
44,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,flashloan_premium_raw,18,raw_token_amount
45,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,flashloan_premium_raw,6,raw_token_amount
46,0x1f84a51296691320478c98b8d77f2bbd17d34350,PT-USDe-5FEB2026,flashloan_premium_raw,18,raw_token_amount
47,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,flashloan_premium_raw,18,raw_token_amount
48,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,flashloan_premium_raw,8,raw_token_amount
49,0x356b8d89c1e1239cbbb9de4815c39a1474d5ba7d,syrupUSDT,flashloan_premium_raw,6,raw_token_amount
50,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,flashloan_premium_raw,18,raw_token_amount
51,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,flashloan_premium_raw,18,raw_token_amount
52,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,flashloan_premium_raw,18,raw_token_amount


In [41]:
flashloan_agg_cols = [
    "flashloan_tx_count",
    "unique_flashloan_initiators",
    "no_open_debt_flashloan_tx_count",
    "variable_flashloan_tx_count",
]

df_1_flashloan = tf.aggregate_by_time_bucket(df_0_flashloan, "time_bucket", flashloan_agg_cols, agg_func='sum')
df_1_flashloan

,time_bucket,flashloan_tx_count,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,variable_flashloan_tx_count
0,2025-04-01 02:00:00.000 UTC,26,20,19,7
1,2025-04-01 04:00:00.000 UTC,23,19,19,4
2,2025-04-01 06:00:00.000 UTC,9,7,8,1
3,2025-04-01 08:00:00.000 UTC,17,7,6,11
4,2025-04-01 10:00:00.000 UTC,28,17,17,11
...,...,...,...,...,...
4356,2026-03-30 16:00:00.000 UTC,21,11,19,2
4357,2026-03-30 18:00:00.000 UTC,11,5,11,0
4358,2026-03-30 20:00:00.000 UTC,20,10,18,2
4359,2026-03-30 22:00:00.000 UTC,9,5,9,0


In [42]:
df_2_flashloan_scaled = tf.scale_by_decimals(df_0_flashloan, df_2_flashloan_amount, drop_raw=True)
df_2_flashloan_scaled.drop(flashloan_agg_cols, axis=1, inplace=True)
df_2_flashloan_scaled.drop(["stable_flashloan_tx_count", "latest_flashloan_block"], axis=1, inplace=True)
df_2_flashloan_scaled_1 = tf.multiply_by_price(df_2_flashloan_scaled, df_oracle_price_usd_eth, ["flashloan_amount", "flashloan_premium"])
# df_3_borrow = tf.aggregate_by_time_bucket(df_2_borrow, "time_bucket", borrow_agg_cols, agg_func='sum')

stats_flashloan = [
    "flashloan_amount_value_usd",
    "flashloan_amount_value_eth",
    "flashloan_premium_value_usd",
    "flashloan_premium_value_eth",
]

df_2_flashloan_scaled_1



,time_bucket,asset,asset_symbol,flashloan_amount,flashloan_premium,flashloan_amount_value_usd,flashloan_amount_value_eth,flashloan_premium_value_usd,flashloan_premium_value_eth
0,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,2.023045e+00,0.001012,1.670107e+05,91.413693,83.505152,0.045707
1,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,2.696462e+01,0.013482,5.900274e+04,32.295284,29.501372,0.016148
2,2025-04-01 02:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,2.397038e+05,87.236780,2.398672e+05,131.291881,87.296257,0.047782
3,2025-04-01 02:00:00.000 UTC,0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f,SNX,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-04-01 02:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,1.899079e+02,0.012555,3.469574e+05,189.907915,22.937446,0.012555
...,...,...,...,...,...,...,...,...,...
24883,2026-03-31 00:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,6.060024e+01,0.030301,6.061352e+01,0.029874,0.030308,0.000015
24884,2026-03-31 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,2.672996e+06,1336.498059,2.673147e+06,1317.486807,1336.573348,0.658743
24885,2026-03-31 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,1.183152e+03,0.591576,2.400606e+06,1183.151681,1200.302944,0.591576
24886,2026-03-31 00:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,1.000028e-02,0.000005,6.669985e+02,0.328736,0.334157,0.000165


In [43]:
df_2_flashloan_scaled_2 = tf.aggregate_by_time_bucket(df_2_flashloan_scaled_1, "time_bucket", stats_flashloan , agg_func='sum')
df_2_flashloan_scaled_2


,time_bucket,flashloan_amount_value_usd,flashloan_amount_value_eth,flashloan_premium_value_usd,flashloan_premium_value_eth
0,2025-04-01 02:00:00.000 UTC,9.411896e+05,515.162277,248.230641,0.135870
1,2025-04-01 04:00:00.000 UTC,6.512491e+06,3548.047308,3038.132814,1.655194
2,2025-04-01 06:00:00.000 UTC,2.605000e+05,141.502472,8.453992,0.004592
3,2025-04-01 08:00:00.000 UTC,1.136226e+06,612.843849,147.875707,0.079760
4,2025-04-01 10:00:00.000 UTC,2.134066e+06,1135.526324,708.085451,0.376769
...,...,...,...,...,...
4356,2026-03-30 16:00:00.000 UTC,1.972011e+05,95.570965,98.315866,0.047648
4357,2026-03-30 18:00:00.000 UTC,3.569976e+06,1734.672485,1784.988586,0.867336
4358,2026-03-30 20:00:00.000 UTC,2.515741e+06,1240.183769,1252.234029,0.617313
4359,2026-03-30 22:00:00.000 UTC,2.691076e+06,1326.053058,1345.538013,0.663027


In [44]:
DF_flashloan_common = df_2_flashloan_scaled_2.merge(
    df_1_flashloan,
      on=["time_bucket"], how="left", suffixes=("_x", "_y"))

DF_flashloan_common

,time_bucket,flashloan_amount_value_usd,flashloan_amount_value_eth,flashloan_premium_value_usd,flashloan_premium_value_eth,flashloan_tx_count,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,variable_flashloan_tx_count
0,2025-04-01 02:00:00.000 UTC,9.411896e+05,515.162277,248.230641,0.135870,26,20,19,7
1,2025-04-01 04:00:00.000 UTC,6.512491e+06,3548.047308,3038.132814,1.655194,23,19,19,4
2,2025-04-01 06:00:00.000 UTC,2.605000e+05,141.502472,8.453992,0.004592,9,7,8,1
3,2025-04-01 08:00:00.000 UTC,1.136226e+06,612.843849,147.875707,0.079760,17,7,6,11
4,2025-04-01 10:00:00.000 UTC,2.134066e+06,1135.526324,708.085451,0.376769,28,17,17,11
...,...,...,...,...,...,...,...,...,...
4356,2026-03-30 16:00:00.000 UTC,1.972011e+05,95.570965,98.315866,0.047648,21,11,19,2
4357,2026-03-30 18:00:00.000 UTC,3.569976e+06,1734.672485,1784.988586,0.867336,11,5,11,0
4358,2026-03-30 20:00:00.000 UTC,2.515741e+06,1240.183769,1252.234029,0.617313,20,10,18,2
4359,2026-03-30 22:00:00.000 UTC,2.691076e+06,1326.053058,1345.538013,0.663027,9,5,9,0


In [45]:
TABLE = "user_account"

df_0_user_account = run.load(TABLE)
sc.check_frame(df_0_user_account, PROCESS_SCHEMAS[TABLE], table=TABLE, stage="transform:input")
print(f"{TABLE}: {df_0_user_account.shape[0]} rows x {df_0_user_account.shape[1]} cols")
display(df_0_user_account.dtypes.to_frame("dtype"))
display(df_0_user_account.head(PREVIEW_ROWS))
df_0_user_account.drop(["min_health_factor", "max_health_factor"], axis=1, inplace=True)

user_account: 2174 rows x 10 cols


,dtype
time_bucket,object
avg_total_collateral_base,float64
avg_total_debt_base,float64
avg_available_borrows_base,float64
avg_current_liquidation_threshold,float64
avg_ltv,float64
min_health_factor,float64
max_health_factor,float64
sampled_user_count,int64
account_data_call_count,int64


,time_bucket,avg_total_collateral_base,avg_total_debt_base,avg_available_borrows_base,avg_current_liquidation_threshold,avg_ltv,min_health_factor,max_health_factor,sampled_user_count,account_data_call_count
0,2025-04-01 04:00:00.000 UTC,1.491521e+16,4.997479e+15,6.710961e+15,8100.0,7850.000000,2.417483e+18,2.417483e+18,1,1
1,2025-04-01 06:00:00.000 UTC,1.506325e+16,5.192650e+15,6.631999e+15,8100.0,7850.000000,2.349712e+18,2.349712e+18,1,1
2,2025-04-01 10:00:00.000 UTC,1.262487e+11,4.772150e+10,4.696503e+10,7800.0,7500.000000,2.063514e+18,2.063514e+18,1,1
3,2025-04-01 14:00:00.000 UTC,1.047713e+16,6.422148e+15,3.321578e+15,9500.0,9300.000000,1.549835e+18,1.549835e+18,1,1
4,2025-04-02 14:00:00.000 UTC,1.047771e+16,6.422465e+15,3.321807e+15,9500.0,9300.000000,1.549845e+18,1.549845e+18,1,1
5,2025-04-03 00:00:00.000 UTC,8.024767e+11,5.746264e+11,2.643676e+10,7793.0,7271.666667,1.004146e+18,1.148862e+18,3,3
6,2025-04-03 12:00:00.000 UTC,1.478903e+16,4.509097e+15,7.100292e+15,8100.0,7850.000000,2.656655e+18,2.656655e+18,1,1
7,2025-04-03 14:00:00.000 UTC,5.965423e+14,3.615911e+14,1.875464e+14,7799.5,7534.055556,1.042772e+18,1.157921e+77,9,18
8,2025-04-03 16:00:00.000 UTC,1.970535e+13,1.336692e+13,1.571601e+12,7985.5,7581.500000,1.153408e+18,1.204179e+18,1,2
9,2025-04-03 22:00:00.000 UTC,6.572538e+13,5.917612e+13,1.948483e+12,9500.0,9300.000000,1.054971e+18,1.055556e+18,1,2


In [46]:
stats_user = [
    df_0_user_account["sampled_user_count"].mean(), 
    df_0_user_account["sampled_user_count"].median(),
    # df_0_user_account["sampled_user_count"].std(),
    # df_0_user_account["sampled_user_count"].var(),
    df_0_user_account["account_data_call_count"].mean(),
    df_0_user_account["account_data_call_count"].median(),
#   df_0_user_account["account_data_call_count"].std(),
#   df_0_user_account["account_data_call_count"].var()
]

print(stats_user)

[2.2594296228150874, 1.0, 6.21343146274149, 3.0]


In [47]:
TABLE = "decimal_reference_part6"

df_0_user_account_agg = run.load(TABLE)
sc.check_frame(df_0_user_account_agg, PROCESS_SCHEMAS[TABLE], table=TABLE, stage="transform:input")
print(f"{TABLE}: {df_0_user_account_agg.shape[0]} rows x {df_0_user_account_agg.shape[1]} cols")
display(df_0_user_account_agg.dtypes.to_frame("dtype"))
# display(df_0_user_account_agg.head(PREVIEW_ROWS))

df_0_user_account_agg_1 = df_0_user_account_agg.drop(df_0_user_account_agg.index[[0, 8]])
df_0_user_account_agg_1


decimal_reference_part6: 9 rows x 5 cols


,dtype
asset,float64
asset_symbol,float64
metric,object
decimals,int64
unit,object


,asset,asset_symbol,metric,decimals,unit
1,NaN,NaN,avg_available_borrows_base,8,usd_base_8dp
2,NaN,NaN,avg_current_liquidation_threshold,4,basis_points
3,NaN,NaN,avg_ltv,4,basis_points
4,NaN,NaN,avg_total_collateral_base,8,usd_base_8dp
5,NaN,NaN,avg_total_debt_base,8,usd_base_8dp
6,NaN,NaN,max_health_factor,18,wad
7,NaN,NaN,min_health_factor,18,wad


In [48]:
metric_cols = dict(zip(
    df_0_user_account_agg_1["metric"],
    df_0_user_account_agg_1["decimals"]
))

    
out = df_0_user_account.copy()

for col, dec in metric_cols.items():
    if col not in out.columns:
        continue  # skip metrics that aren't columns in this frame
    out[col] = out[col].astype("float64") / (10 ** int(dec))
df_0_user_account_agg_scaled = out
df_0_user_account_agg_scaled

,time_bucket,avg_total_collateral_base,avg_total_debt_base,avg_available_borrows_base,avg_current_liquidation_threshold,avg_ltv,sampled_user_count,account_data_call_count
0,2025-04-01 04:00:00.000 UTC,1.491521e+08,4.997479e+07,6.710961e+07,0.810,0.785,1,1
1,2025-04-01 06:00:00.000 UTC,1.506325e+08,5.192650e+07,6.631999e+07,0.810,0.785,1,1
2,2025-04-01 10:00:00.000 UTC,1.262487e+03,4.772150e+02,4.696503e+02,0.780,0.750,1,1
3,2025-04-01 14:00:00.000 UTC,1.047713e+08,6.422148e+07,3.321578e+07,0.950,0.930,1,1
4,2025-04-02 14:00:00.000 UTC,1.047771e+08,6.422465e+07,3.321807e+07,0.950,0.930,1,1
...,...,...,...,...,...,...,...,...
2169,2026-03-30 00:00:00.000 UTC,8.784238e+04,6.585792e+04,4.849285e+03,0.742,0.719,6,10
2170,2026-03-30 06:00:00.000 UTC,3.156980e+04,2.259402e+04,2.819663e+03,0.830,0.805,1,2
2171,2026-03-30 12:00:00.000 UTC,1.539400e+06,1.129746e+06,2.557143e+05,0.930,0.900,1,2
2172,2026-03-30 18:00:00.000 UTC,7.581365e+00,3.864152e+00,3.186517e+00,0.950,0.930,1,1


In [49]:
temp_stats = adv.statistical_validation(df_0_user_account_agg_scaled, save = False)

display(temp_stats[["column", "null_pct", "zero_pct", "negative_pct",]])
temp_cols = df_0_user_account_agg_scaled.columns

,column,null_pct,zero_pct,negative_pct
0,avg_total_collateral_base,0.0,1.6559,0.0
1,avg_total_debt_base,0.0,8.0957,0.0
2,avg_available_borrows_base,0.0,3.3579,0.0
3,avg_current_liquidation_threshold,0.0,1.6559,0.0
4,avg_ltv,0.0,1.7479,0.0
5,sampled_user_count,0.0,0.0000,0.0
6,account_data_call_count,0.0,0.0000,0.0


In [50]:
# Separate coarse-grain panels for the sparse liquidation & user-account series, kept
# out of the 2h DF_common_final: liquidations are ~72% empty per 2h bucket, so rolling
# to 24h/7d densifies them. Liquidation cols SUM; user avg_* state MEAN, counts SUM.
from data_validation import canonicalize_keys

user_mean_cols = [
    "avg_total_collateral_base", "avg_total_debt_base", "avg_available_borrows_base",
    "avg_current_liquidation_threshold", "avg_ltv",
]
user_sum_cols = ["sampled_user_count", "account_data_call_count"]
user_all_cols = user_mean_cols + user_sum_cols
user_agg_map = {**{c: "mean" for c in user_mean_cols}, **{c: "sum" for c in user_sum_cols}}

liq_2h  = canonicalize_keys(df_4_liq)
liq_24h = canonicalize_keys(tf.aggregate_by_time_bucket(df_4_liq, "time_bucket", liq_agg_cols, agg_func="sum", freq="24h"))
liq_7d  = canonicalize_keys(tf.aggregate_by_time_bucket(df_4_liq, "time_bucket", liq_agg_cols, agg_func="sum", freq="7d"))

user_2h  = canonicalize_keys(df_0_user_account_agg_scaled)
user_24h = canonicalize_keys(tf.aggregate_by_time_bucket(df_0_user_account_agg_scaled, "time_bucket", user_all_cols, agg_func=user_agg_map, freq="24h"))
user_7d  = canonicalize_keys(tf.aggregate_by_time_bucket(df_0_user_account_agg_scaled, "time_bucket", user_all_cols, agg_func=user_agg_map, freq="7d"))

print("liq  2h/24h/7d :", liq_2h.shape, liq_24h.shape, liq_7d.shape)
print("user 2h/24h/7d :", user_2h.shape, user_24h.shape, user_7d.shape)

liq  2h/24h/7d : (1222, 10) (365, 10) (53, 10)
user 2h/24h/7d : (2174, 8) (364, 8) (52, 8)


In [51]:
# df_4_supply_withdraw
# df_3_borrow
# DF_common_1
# df_4_liq
# df_1_collateral
# DF_flashloan_common
# df_0_user_account_agg_scaled

DF_common_final_1 = df_4_supply_withdraw.merge(
    df_3_borrow, on=["time_bucket"], how="left", suffixes=("", "_y")).merge(
    # df_4_liq, on=["time_bucket"], how="left", suffixes=("", "_liq")).merge(
    df_1_collateral, on=["time_bucket"], how="left", suffixes=("", "_collateral")).merge(
    DF_flashloan_common, on=["time_bucket"], how="left", suffixes=("", "_flashloan"))
    # df_0_user_account_agg_scaled, on=["time_bucket"], how="left", suffixes=("", "_useragg"))

display(DF_common_final_1.head(PREVIEW_ROWS))
DF_common_final_1
display(DF_common_final_1.dtypes.to_frame("dtype"))
display(DF_common_1.head(PREVIEW_ROWS))


,time_bucket,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth,borrow_tx_count,...,unique_collateral_enable_users,unique_collateral_disable_users,flashloan_amount_value_usd,flashloan_amount_value_eth,flashloan_premium_value_usd,flashloan_premium_value_eth,flashloan_tx_count,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,variable_flashloan_tx_count
0,2025-04-01 02:00:00.000 UTC,107,112,54,60,1.814934e+08,99340.849932,9.766776e+07,53458.667287,54.0,...,59.0,56.0,9.411896e+05,515.162277,248.230641,0.135870,26.0,20.0,19.0,7.0
1,2025-04-01 04:00:00.000 UTC,116,105,60,55,2.512122e+08,136862.261136,2.665668e+08,145227.537741,58.0,...,61.0,41.0,6.512491e+06,3548.047308,3038.132814,1.655194,23.0,19.0,19.0,4.0
2,2025-04-01 06:00:00.000 UTC,72,100,38,54,1.096440e+08,59557.933326,1.329771e+08,72232.382778,48.0,...,41.0,36.0,2.605000e+05,141.502472,8.453992,0.004592,9.0,7.0,8.0,1.0
3,2025-04-01 08:00:00.000 UTC,129,115,58,48,6.726555e+08,362808.719631,6.691233e+08,360903.546607,64.0,...,58.0,39.0,1.136226e+06,612.843849,147.875707,0.079760,17.0,7.0,6.0,11.0
4,2025-04-01 10:00:00.000 UTC,138,118,68,53,3.680422e+08,195833.482106,3.695117e+08,196615.376327,95.0,...,68.0,43.0,2.134066e+06,1135.526324,708.085451,0.376769,28.0,17.0,17.0,11.0
5,2025-04-01 12:00:00.000 UTC,127,111,77,59,2.532615e+08,135086.519080,2.552678e+08,136156.675648,72.0,...,62.0,57.0,2.762158e+06,1473.313495,235.541791,0.125635,15.0,12.0,11.0,4.0
6,2025-04-01 14:00:00.000 UTC,127,122,60,62,4.748889e+08,254080.767024,4.601955e+08,246219.340371,57.0,...,63.0,58.0,9.532554e+05,510.022369,194.910465,0.104283,17.0,12.0,14.0,3.0
7,2025-04-01 16:00:00.000 UTC,142,111,66,60,5.869603e+08,311246.523260,4.781272e+08,253535.842057,98.0,...,61.0,51.0,1.710213e+06,906.836451,400.869811,0.212556,36.0,21.0,17.0,19.0
8,2025-04-01 18:00:00.000 UTC,122,103,62,55,1.321209e+08,68913.296488,1.362937e+08,71089.779751,82.0,...,49.0,44.0,4.495563e+05,234.485268,68.061213,0.035500,24.0,13.0,11.0,13.0
9,2025-04-01 20:00:00.000 UTC,137,98,54,55,5.251323e+08,275222.083697,5.202348e+08,272655.318074,97.0,...,46.0,35.0,4.160916e+05,218.073803,89.506585,0.046910,30.0,16.0,25.0,5.0


,dtype
time_bucket,object
supply_tx_count,int64
withdrawal_tx_count,int64
unique_suppliers,int64
unique_withdraw_users,int64
supply_amount_value_usd,float64
supply_amount_value_eth,float64
withdrawal_amount_value_usd,float64
withdrawal_amount_value_eth,float64
borrow_tx_count,float64


,time_bucket,asset,asset_symbol,last_borrow_rate,liquidity_rate,variable_borrow_rate,liquidity_index,variable_borrow_index,update_count_liquidity
0,2025-04-01 02:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,0.005103,0.000260,0.005101,1.003301,1.022170,21
1,2025-04-01 02:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0.045,0.000000,0.045000,1.000000,1.123769,5
2,2025-04-01 02:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,0.033222,0.011037,0.033222,1.038794,1.074808,5
3,2025-04-01 02:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,0.048095,0.026601,0.048095,1.121824,1.179228,2
4,2025-04-01 02:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,0.004968,0.001172,0.004968,1.001093,1.009286,6
5,2025-04-01 02:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,0.046031,0.029176,0.045980,1.125029,1.164874,91
6,2025-04-01 02:00:00.000 UTC,0xc011a73ee8576fb46f5e1c5751ca3b9fe0af2a6f,SNX,0.089417,0.017815,0.088239,1.023239,1.127670,3
7,2025-04-01 02:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,0.026735,0.020251,0.026735,1.042603,1.067234,96
8,2025-04-01 02:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,0.04826,0.032141,0.048260,1.119375,1.163539,73
9,2025-04-01 02:00:00.000 UTC,0xf939e0a03fb07f59a73314e73794be0e57ac1b4e,crvUSD,NaN,0.035663,0.054515,1.144648,1.189138,1


In [ ]:
# ── common-window trim (ERR-03): oracle prices end before the event extracts, so any
# bucket past the last priced one values real events at $0 — the model target included.
# The cutoff is rebuilt in canonical "....000 UTC" form so the string compares align.
_cut_ts = pd.to_datetime(df_oracle_price_usd_eth["time_bucket"].max())
CUTOFF = _cut_ts.strftime("%Y-%m-%d %H:%M:%S.000 UTC")
CUTOFF_7D = _cut_ts 

def _trim_2h(df):
    return df.loc[df["time_bucket"] <= CUTOFF].reset_index(drop=True)

def _trim_7d(df):
    days = pd.to_datetime(df["time_bucket"].str.replace(" UTC", "", regex=False))
    return df.loc[days <= CUTOFF_7D].reset_index(drop=True)

DF_common_final_1 = _trim_2h(DF_common_final_1)
DF_common_1 = _trim_2h(DF_common_1)
liq_2h, user_2h = _trim_2h(liq_2h), _trim_2h(user_2h)
liq_24h, user_24h = _trim_2h(liq_24h), _trim_2h(user_24h)   # 24h bucket on the cutoff day is fully priced
liq_7d, user_7d = _trim_7d(liq_7d), _trim_7d(user_7d)       # a 7d bucket needs 7 priced days

# ── E27 null semantics (ERR-04): every DF_common_final column is an EVENT aggregate,
# so a quiet bucket is 0, never null. Guarded so an unrepaired frame can't be written.
event_cols = [c for c in DF_common_final_1.columns if c != "time_bucket"]
DF_common_final_1 = tf.fill_event_zeros(DF_common_final_1, event_cols)
assert DF_common_final_1[event_cols].notna().all().all(), "nulls left in event columns"
assert DF_common_final_1["time_bucket"].max() == CUTOFF, "trim missed the oracle cutoff"

# ── sampled user STATE carried forward max 2 days WITH audit metadata (ERR-17);
# call counts are events -> 0 on no-call days, never forward-filled.
user_24h = tf.fill_event_zeros(user_24h, ["sampled_user_count", "account_data_call_count"])
user_24h = tf.ffill_state_columns(
    user_24h, user_mean_cols, limit=2,
    observed_col="user_state_observed", age_col="user_state_age")

# ── save. Every frame is asked what window it holds (tf.month_span / tf.date_span) and the
# answer becomes the filename; the FOLDER is the run's own timeline, so an export always
# lands under the window it was produced from: transformed_data/batch_<start_month>_<end_month>/
# <table>_<start_date>_<end_date>.csv. Two windows of exports can now sit side by side without
# either one being "the" DF_common_final. All naming/placement lives in tf.save_batch.
# The 2h panel and the 24h/7d rollups are one batch: same window, different grain — and the
# 7d frames legitimately end ~6 days earlier (CUTOFF_7D), which their filenames state.
_out_frames = {
    "DF_common_final_1":     DF_common_final_1,
    "DF_common_1":         DF_common_1,
    # separate liquidation & user-account panels (2h as-is + 24h / 7d rollups) — kept
    # OUT of DF_common_final; feature_addition builds the sparse-family features on these.
    "df_liq_2h":           liq_2h,
    "df_liq_24h":          liq_24h,
    "df_liq_7d":           liq_7d,
    "df_user_account_2h":  user_2h,
    "df_user_account_24h": user_24h,
    "df_user_account_7d":  user_7d,
}
# Output gate: the structure this notebook promises downstream. It runs BEFORE the write, so
# a drifted frame never reaches disk for feature_addition to inherit.
sc.check_frames(_out_frames, TRANSFORM_SCHEMAS, stage="transform:output")
saved = run.save(_out_frames)
display(saved)

# per-stage intermediates, same scheme, under processed_data/ — named after the table each
# one holds rather than the old table_<i>.csv, whose number said nothing about its contents
# and shifted whenever the list was reordered.
_processed_frames = {
    "supply_withdraw":   _trim_2h(df_4_supply_withdraw),
    "borrow_repay":      _trim_2h(df_3_borrow),
    "reserve_panel":     DF_common_1,                       # already trimmed above
    "liquidation":       _trim_2h(df_4_liq),
    "collateral_toggle": _trim_2h(df_1_collateral),
    "flashloan":         _trim_2h(DF_flashloan_common),
    "user_account":      _trim_2h(df_0_user_account_agg_scaled),
}
sc.check_frames(_processed_frames, TRANSFORM_SCHEMAS, stage="transform:output")
# stage="transform" nests these one level deeper — processed_data/batch_<span>/transform/ —
# so a second stage writing per-table intermediates here cannot collide with these seven.
saved_processed = run.save(_processed_frames, out_dir="processed_data", stage="transform")
display(saved_processed)

# what this run actually opened — a file absent here was never read by this run.
display(run.loaded())

print(f"trimmed to priced window: last bucket {CUTOFF}")
# The 12 input gates above are silent on pass (a passing assert prints nothing), so
# say so once here — a gate nobody can see having run is a gate nobody trusts.
print(f"schema gates: {len(run.used)} raw tables checked on load, "
      f"{len(_out_frames)} + {len(_processed_frames)} checked before write")


[transform:output] schema OK: 8/15 tables match; 7 not produced by this run (borrow_repay, collateral_toggle, flashloan, liquidation, reserve_panel, supply_withdraw, user_account)
wrote 8 tables -> transformed_data/batch_2025-04_2026-03/


,table,file,rows,cols,first_date,last_date,span,matches_batch,superseded
0,DF_common_final_1,DF_common_final_1_2025-04-01_2026-03-31.csv,4368,30,2025-04-01,2026-03-31,2025-04 → 2026-03,True,DF_common_final_1_2025-04-01_2026-03-30.csv
1,DF_common_1,DF_common_1_2025-04-01_2026-03-31.csv,54198,9,2025-04-01,2026-03-31,2025-04 → 2026-03,True,DF_common_1_2025-04-01_2026-03-30.csv
2,df_liq_2h,df_liq_2h_2025-04-01_2026-03-31.csv,1222,10,2025-04-01,2026-03-31,2025-04 → 2026-03,True,df_liq_2h_2025-04-01_2026-03-30.csv
3,df_liq_24h,df_liq_24h_2025-04-01_2026-03-31.csv,365,10,2025-04-01,2026-03-31,2025-04 → 2026-03,True,df_liq_24h_2025-04-01_2026-03-30.csv
4,df_liq_7d,df_liq_7d_2025-04-01_2026-03-24.csv,52,10,2025-04-01,2026-03-24,2025-04 → 2026-03,True,
5,df_user_account_2h,df_user_account_2h_2025-04-01_2026-03-30.csv,2174,8,2025-04-01,2026-03-30,2025-04 → 2026-03,True,
6,df_user_account_24h,df_user_account_24h_2025-04-01_2026-03-30.csv,364,10,2025-04-01,2026-03-30,2025-04 → 2026-03,True,
7,df_user_account_7d,df_user_account_7d_2025-04-01_2026-03-24.csv,52,8,2025-04-01,2026-03-24,2025-04 → 2026-03,True,


[transform:output] schema OK: 7/15 tables match; 8 not produced by this run (DF_common_1, DF_common_final_1, df_liq_24h, df_liq_2h, df_liq_7d, df_user_account_24h, df_user_account_2h, df_user_account_7d)
wrote 7 tables -> processed_data/batch_2025-04_2026-03/transform/


,table,file,rows,cols,first_date,last_date,span,matches_batch,superseded
0,supply_withdraw,supply_withdraw_2025-04-01_2026-03-31.csv,4368,9,2025-04-01,2026-03-31,2025-04 → 2026-03,True,supply_withdraw_2025-04-01_2026-03-30.csv
1,borrow_repay,borrow_repay_2025-04-01_2026-03-31.csv,4368,10,2025-04-01,2026-03-31,2025-04 → 2026-03,True,borrow_repay_2025-04-01_2026-03-30.csv
2,reserve_panel,reserve_panel_2025-04-01_2026-03-31.csv,54198,9,2025-04-01,2026-03-31,2025-04 → 2026-03,True,reserve_panel_2025-04-01_2026-03-30.csv
3,liquidation,liquidation_2025-04-01_2026-03-31.csv,1222,10,2025-04-01,2026-03-31,2025-04 → 2026-03,True,liquidation_2025-04-01_2026-03-30.csv
4,collateral_toggle,collateral_toggle_2025-04-01_2026-03-31.csv,4368,5,2025-04-01,2026-03-31,2025-04 → 2026-03,True,collateral_toggle_2025-04-01_2026-03-30.csv
5,flashloan,flashloan_2025-04-01_2026-03-31.csv,4361,9,2025-04-01,2026-03-31,2025-04 → 2026-03,True,flashloan_2025-04-01_2026-03-30.csv
6,user_account,user_account_2025-04-01_2026-03-30.csv,2174,8,2025-04-01,2026-03-30,2025-04 → 2026-03,True,


,table,file,rows,cols
0,supply_withdraw,supply_withdraw_8595322_2025-04-01_2026-03-31.csv,84609,12
1,oracle_price_usd_eth_weth_6h,oracle_price_usd_eth_weth_6h_8403530_2025-04-0...,206330,7
2,borrow_repay,borrow_repay_7798273_2025-04-01_2026-03-31.csv,54198,15
3,reserve_state_rates,reserve_state_rates_7711042_2025-04-01_2026-03...,99143,8
4,reserve_config,reserve_config_7804264_asof_2026-03-31.csv,232175,11
5,liquidation,liquidation_7798339_2025-04-01_2026-03-31.csv,4468,12
6,decimal_reference_part4,decimal_reference_part4_7711290_20260623T19231...,82,5
7,collateral_toggle,collateral_toggle_7798372_2025-04-01_2026-03-3...,60436,8
8,flashloan,flashloan_7798349_2025-04-01_2026-03-31.csv,24888,11
9,decimal_reference_part5,decimal_reference_part5_7711298_20260623T19232...,95,5


trimmed to priced window: last bucket 2026-03-31 00:00:00.000 UTC
schema gates: 12 raw tables checked on load, 8 + 7 checked before write
